# Lesson 00: Setup and tooling (solution)

**Goal:** start the project and watch the whole fake pipeline answer a question about `tiny.pdf`.

**Before you start**
- Theory: [lesson 00](../../../docs/lessons/00_setup_and_tooling.md)
- Python concepts: packages vs. modules, `importlib.metadata`, top-level `await` in Jupyter
- Files used: `configs/e2e_fake.yaml`, `fixtures/tiny.pdf`. Offline, no API key, no cost.

Each step: read the short explanation, fill in the `# TODO` cell, run the check cell below it.

## Setup (just run it)

In [ ]:
import warnings
from importlib.metadata import version
from pathlib import Path

import docrag
from docrag.ingest.indexer import index_documents
from docrag.shared.config import ExperimentConfig
from docrag.shared.settings import Settings
from docrag.wiring import build_answer_service

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
warnings.filterwarnings(
    "ignore", message="Payload indexes have no effect"
)  # in-memory Qdrant
print("docrag is imported from", Path(docrag.__file__).parent)

## Step 1: Which version is installed?

The **distribution** is called `docrag-bench` (the name in `pyproject.toml`, like an npm package name), the **import package** is `docrag` (the folder in `src/`). `importlib.metadata.version(name)` reads the installed distribution's version, the way `require("pkg/package.json").version` would in Node.

In [ ]:
docrag_version = version("docrag-bench")

In [ ]:
assert docrag_version is not None, "Hint: version('docrag-bench')"
assert docrag_version.count(".") == 2, (
    f"Expected something like 0.1.0, got {docrag_version!r}"
)
print("docrag-bench", docrag_version)
print("OK")

## Step 2: Load an experiment config

Two kinds of configuration exist on purpose: `Settings` (paths, URLs, API keys, from the environment) and `ExperimentConfig` (what is measured, one YAML under `configs/`). `ExperimentConfig.from_yaml(path)` is an alternative constructor (a `@classmethod`, like a static factory in Java).

In [ ]:
config = ExperimentConfig.from_yaml(ROOT / "configs" / "e2e_fake.yaml")

In [ ]:
assert config is not None, (
    "Hint: ExperimentConfig.from_yaml(ROOT / 'configs' / 'e2e_fake.yaml')"
)
assert config.llm == "fake:echo" and config.embedder == "fake:64", (
    "e2e_fake uses only fakes"
)
assert len(config.config_hash()) == 12, "The hash identifies WHAT is measured"
# Settings: Qdrant in-process (":memory:"), no OS keyring lookup.
settings = Settings(qdrant_location=":memory:", use_keyring=False)
print(config.name, config.config_hash(), "| qdrant:", settings.qdrant_location)
print("OK")

## Step 3: The whole pipeline, with fakes

The next cell builds the real object graph (`wiring.py`) and indexes `tiny.pdf`. Then you ask a question. `service.answer` is `async`: in a script you'd need `asyncio.run(...)`, but Jupyter already runs an event loop, so you can write `await` directly in a cell.

In [ ]:
service = build_answer_service(settings, config, cache=None)
added = index_documents(
    {"tiny": ROOT / "fixtures" / "tiny.pdf"},
    service.retriever.store,
    service.retriever.embeddings,
    prefix_headings=config.prefix_headings,
    cache=None,
    workers=1,
)
print("indexed", added, "chunks")

In [ ]:
question = "What was the revenue in Europe in 2024?"
result, hits = await service.answer(question, doc_ids=["tiny"])

In [ ]:
assert result is not None, (
    "Hint: result, hits = await service.answer(question, doc_ids=['tiny'])"
)
assert result.status == "supported", f"Expected a supported answer, got {result.status}"
assert result.verified_citations, (
    "The fake LLM cites a real quote, and Verify accepts it"
)
print(result.answer)
for c in result.citations:
    print(" ", "OK " if c.verified else "REJ", c.id, repr(c.quote))
await service.proxy.aclose()
print("OK")

## Observation question

Why does all of this run without an API key, a GPU or Docker?

<details>
<summary>Answer</summary>

Every component in `e2e_fake.yaml` is a fake with the same interface as the real one: `fake:64` hashes words into a 64-dim vector, `fake:overlap` reranks by word overlap, and the LLM route `fake:echo` resolves to `FakeProvider`, which `LLMProxy.from_settings` always registers. It answers by citing the first context passage with a real quote. Qdrant runs in-process (`":memory:"`). Paid providers are only registered when their key exists, so a route to one would fail with a clear message instead of a silent call. The same fakes drive CI and the E2E test.
</details>

## The real code

- Composition root: [`src/docrag/wiring.py`](../../../src/docrag/wiring.py)
- CLI: [`src/docrag/cli.py`](../../../src/docrag/cli.py)
- Fake LLM: [`src/docrag/llm/fake.py`](../../../src/docrag/llm/fake.py)
- Lesson: [`docs/lessons/00_setup_and_tooling.md`](../../../docs/lessons/00_setup_and_tooling.md)